In [1]:
import os

BASE_PATH = ".."
RAW_PATH = os.path.join(BASE_PATH, "raw")
PROCESSED_PATH = os.path.join(BASE_PATH, "processed")

print("Raw path:", RAW_PATH)
print("Archivos en raw:")
print(os.listdir(RAW_PATH))

Raw path: ../raw
Archivos en raw:
['tracks.csv', 'dict_artists.json', 'artists.csv']


In [2]:
# Iniciamos sesión de Spark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HitMakerAI") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 20:42:48 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp3s0)
26/05/03 20:42:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 20:42:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Antes que nada. Primero veamos las columnas de cada .csv

In [3]:
tracks_header = spark.sparkContext.textFile("../raw/tracks.csv").first()
artists_header = spark.sparkContext.textFile("../raw/artists.csv").first()

print("Columnas de tracks.csv:")
print(tracks_header.split(","))

print("\nColumnas de artists.csv:")
print(artists_header.split(","))

Columnas de tracks.csv:
['id', 'name', 'popularity', 'duration_ms', 'explicit', 'artists', 'id_artists', 'release_date', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']

Columnas de artists.csv:
['id', 'followers', 'genres', 'name', 'popularity']


Como el archivo es grande, conviene no usar $inferSchema=True$, porque Spark tendría que escanear mucho para adivinar tipos.

Mejor definimos el esquema manualmente.

In [4]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

In [5]:
tracks_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("popularity", IntegerType(), True),
    StructField("duration_ms", IntegerType(), True),
    StructField("explicit", IntegerType(), True),
    StructField("artists", StringType(), True),
    StructField("id_artists", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("danceability", DoubleType(), True),
    StructField("energy", DoubleType(), True),
    StructField("key", IntegerType(), True),
    StructField("loudness", DoubleType(), True),
    StructField("mode", IntegerType(), True),
    StructField("speechiness", DoubleType(), True),
    StructField("acousticness", DoubleType(), True),
    StructField("instrumentalness", DoubleType(), True),
    StructField("liveness", DoubleType(), True),
    StructField("valence", DoubleType(), True),
    StructField("tempo", DoubleType(), True),
    StructField("time_signature", IntegerType(), True)
])

artists_schema = StructType([
    StructField("id", StringType(), True),
    StructField("followers", DoubleType(), True),
    StructField("genres", StringType(), True),
    StructField("name", StringType(), True),
    StructField("popularity", IntegerType(), True)
])

Leemos los archivos:

In [6]:
tracks_df = (
    spark.read
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .schema(tracks_schema)
    .csv("../raw/tracks.csv")
)

artists_df = (
    spark.read
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .schema(artists_schema)
    .csv("../raw/artists.csv")
)

Verificamos que haya cargado correctamente:

In [7]:
tracks_df.printSchema()
artists_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)

root
 |-- id: string (nullable = true)
 |-- followers: double (nullable = true)
 |-- genres: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer

Revisamos con algunas columnas

In [8]:
tracks_df.select(
    "id",
    "name",
    "popularity",
    "artists",
    "release_date",
    "danceability",
    "energy",
    "tempo"
).show(10, truncate=False)

+----------------------+-----------------------------------+----------+-------------------+------------+------------+------+-------+
|id                    |name                               |popularity|artists            |release_date|danceability|energy|tempo  |
+----------------------+-----------------------------------+----------+-------------------+------------+------------+------+-------+
|35iwgR4jXetI318WEWsa1Q|Carve                              |6         |['Uli']            |1922-02-22  |0.645       |0.445 |104.851|
|021ht4sdgPcrDgSk7JTbKY|Capítulo 2.16 - Banquero Anarquista|0         |['Fernando Pessoa']|1922-06-01  |0.695       |0.263 |102.009|
|07A5yehtSnoedViJAZkNnc|Vivo para Quererte - Remasterizado |0         |['Ignacio Corsini']|1922-03-21  |0.434       |0.177 |130.418|
|08FmqUhxtyLTn6pAh6bk45|El Prisionero - Remasterizado      |0         |['Ignacio Corsini']|1922-03-21  |0.321       |0.0946|169.98 |
|08y9GfoqCWfOGsKdwojr5e|Lady of the Evening                |0        

In [9]:
#Revisamos el tamaño del dataset
print("Número de canciones:", tracks_df.count())
print("Número de artistas:", artists_df.count())

Número de canciones: 586672
Número de artistas: 1162095


Ahora para poder trabajar con el Dataset de forma más fácil lo pasamos a parquet en la carpeta "/processed" que es la carpeta de nuestros datos procesados

In [10]:
tracks_df.write.mode("overwrite").parquet("../processed/tracks_raw.parquet")

artists_df.write.mode("overwrite").parquet("../processed/artists_raw.parquet")

26/05/03 20:42:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/03 20:43:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

leemos y verificamos:

In [11]:
tracks_raw = spark.read.parquet("../processed/tracks_raw.parquet")
artists_raw = spark.read.parquet("../processed/artists_raw.parquet")

In [12]:
tracks_raw.printSchema()
artists_raw.printSchema()

print("Canciones:", tracks_raw.count())
print("Artistas:", artists_raw.count())

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)

root
 |-- id: string (nullable = true)
 |-- followers: double (nullable = true)
 |-- genres: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer

## Limpieza de los Datos

### Revisión de Nulos:

In [13]:
from pyspark.sql.functions import col, sum as spark_sum

In [14]:
nulls_list = []

for c in tracks_raw.columns:
    null_count = tracks_raw.filter(col(c).isNull()).count()
    nulls_list.append((c, null_count))

nulls_df = spark.createDataFrame(nulls_list, ["column", "null_count"])
nulls_df.orderBy(col("null_count").desc()).show(truncate=False)

+----------------+----------+
|column          |null_count|
+----------------+----------+
|name            |71        |
|mode            |0         |
|id_artists      |0         |
|speechiness     |0         |
|key             |0         |
|release_date    |0         |
|popularity      |0         |
|danceability    |0         |
|loudness        |0         |
|energy          |0         |
|explicit        |0         |
|duration_ms     |0         |
|liveness        |0         |
|artists         |0         |
|acousticness    |0         |
|valence         |0         |
|id              |0         |
|tempo           |0         |
|instrumentalness|0         |
|time_signature  |0         |
+----------------+----------+



Notemos que hat 71 valores nulos en name

### Revisión de Duplicados

In [15]:
total_rows = tracks_raw.count()
unique_ids = tracks_raw.select("id").distinct().count()

print("Filas totales:", total_rows)
print("IDs únicos:", unique_ids)
print("Duplicados por id:", total_rows - unique_ids)

[Stage 83:>                                                         (0 + 8) / 8]

Filas totales: 586672
IDs únicos: 586672
Duplicados por id: 0


por lo que no hay duplicados por canción 

### Revisión de Rangos Numéricos

In [16]:
from pyspark.sql.functions import min as spark_min, max as spark_max, lit

numeric_cols = [
    "popularity",
    "duration_ms",
    "explicit",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature"
]

range_dfs = []

for c in numeric_cols:
    range_df = tracks_raw.select(
        lit(c).alias("columna"),
        spark_min(c).alias("minimo"),
        spark_max(c).alias("maximo")
    )
    range_dfs.append(range_df)

ranges_df = range_dfs[0]

for df in range_dfs[1:]:
    ranges_df = ranges_df.unionByName(df)

ranges_df.show(truncate=False)

[Stage 102:============>    (6 + 2) / 8][Stage 103:==>              (1 + 6) / 8]

+----------------+------+---------+
|columna         |minimo|maximo   |
+----------------+------+---------+
|popularity      |0.0   |100.0    |
|duration_ms     |3344.0|5621218.0|
|explicit        |0.0   |1.0      |
|danceability    |0.0   |0.991    |
|energy          |0.0   |1.0      |
|key             |0.0   |11.0     |
|loudness        |-60.0 |5.376    |
|mode            |0.0   |1.0      |
|speechiness     |0.0   |0.971    |
|acousticness    |0.0   |0.996    |
|instrumentalness|0.0   |1.0      |
|liveness        |0.0   |1.0      |
|valence         |0.0   |1.0      |
|tempo           |0.0   |246.381  |
|time_signature  |0.0   |5.0      |
+----------------+------+---------+



Notemos que time_signature_min= 0.0 y tempo_min=0.0 está raro. lo demás se ve dentro de un rango "aceptable". chequemos cuantas canciones se eliminarían poniendo restricciones:

In [17]:
from pyspark.sql.functions import col, sum as spark_sum, lit

checks = [
    ("name_null", col("name").isNull()),
    ("tempo_invalid", col("tempo") <= 0),
    ("time_signature_invalid", col("time_signature") <= 0),
    ("duration_invalid", col("duration_ms") <= 0),
    ("popularity_invalid", (col("popularity") < 0) | (col("popularity") > 100)),
    ("danceability_invalid", (col("danceability") < 0) | (col("danceability") > 1)),
    ("energy_invalid", (col("energy") < 0) | (col("energy") > 1)),
    ("speechiness_invalid", (col("speechiness") < 0) | (col("speechiness") > 1)),
    ("acousticness_invalid", (col("acousticness") < 0) | (col("acousticness") > 1)),
    ("instrumentalness_invalid", (col("instrumentalness") < 0) | (col("instrumentalness") > 1)),
    ("liveness_invalid", (col("liveness") < 0) | (col("liveness") > 1)),
    ("valence_invalid", (col("valence") < 0) | (col("valence") > 1)),
]

quality_dfs = []

for check_name, condition in checks:
    check_df = tracks_raw.select(
        lit(check_name).alias("validacion"),
        spark_sum(condition.cast("int")).alias("cantidad")
    )
    quality_dfs.append(check_df)

quality_checks_columnar = quality_dfs[0]

for df in quality_dfs[1:]:
    quality_checks_columnar = quality_checks_columnar.unionByName(df)

quality_checks_columnar.show(truncate=False)

[Stage 129:=> (4 + 4) / 8][Stage 130:>  (0 + 4) / 8][Stage 131:>  (0 + 0) / 8]

+------------------------+--------+
|validacion              |cantidad|
+------------------------+--------+
|name_null               |71      |
|tempo_invalid           |328     |
|time_signature_invalid  |337     |
|duration_invalid        |0       |
|popularity_invalid      |0       |
|danceability_invalid    |0       |
|energy_invalid          |0       |
|speechiness_invalid     |0       |
|acousticness_invalid    |0       |
|instrumentalness_invalid|0       |
|liveness_invalid        |0       |
|valence_invalid         |0       |
+------------------------+--------+



según la tabla anterior el problema está en:
- name_null                  71
- tempo_invalid              328
- time_signature_invalid     337

Entonces la limpieza debe enfocarse más en:
1. quitar canciones sin name,
2. quitar canciones con tempo <= 0,
3. quitar canciones con time_signature <= 0,
4. quitar duplicados por id, por buena práctica.

In [18]:
from pyspark.sql.functions import col

invalid_condition = (
    col("name").isNull() |
    (col("tempo") <= 0) |
    (col("time_signature") <= 0)
)

invalid_rows = tracks_raw.filter(invalid_condition)

print("Filas inválidas únicas:", invalid_rows.count())

Filas inválidas únicas: 408


In [19]:
#Veamos algunos ejemplos
invalid_rows.select(
    "id",
    "name",
    "artists",
    "tempo",
    "time_signature",
    "popularity"
).show(20)

+--------------------+--------------------+--------------------+-----+--------------+----------+
|                  id|                name|             artists|tempo|time_signature|popularity|
+--------------------+--------------------+--------------------+-----+--------------+----------+
|5NR2UScCobzDg4zpb...|Sultry Ambiance f...|        ['Cafe BGM']|  0.0|             0|        16|
|035fB4qKVUc0GOW0o...|   Clean White Noise|['White Noise The...|  0.0|             0|        59|
|1RliEhshETEeAcCmA...|     Sleepless Years|['atrabilis sunri...|  0.0|             0|         0|
|7dTyZl4cBRKnPBeq1...|         White Noise|     ['Rain Sounds']|  0.0|             0|        64|
|575DZcL4PtBm3qoFd...|       Box Fan Sound|['Tmsoft’s White ...|  0.0|             0|        68|
|67iF3DdebmElASIGo...|Brown Noise - Loo...|['White Noise Med...|  0.0|             0|        72|
|1C0h4SvAveezqMSeA...|The Early Morning...|['White Noise Bab...|  0.0|             0|        68|
|3hJJOnsafnWFrhWDw...|  Box Fa

### Creación de DataSet (tracks) limpio:

In [20]:
tracks_clean = tracks_raw.filter(
    col("name").isNotNull() &
    (col("tempo") > 0) &
    (col("time_signature") > 0)
)

Verificación:

In [21]:
from pyspark.sql.functions import sum as spark_sum

tracks_clean.select(
    spark_sum(col("name").isNull().cast("int")).alias("name_null"),
    spark_sum((col("tempo") <= 0).cast("int")).alias("tempo_invalid"),
    spark_sum((col("time_signature") <= 0).cast("int")).alias("time_signature_invalid")
).show()

+---------+-------------+----------------------+
|name_null|tempo_invalid|time_signature_invalid|
+---------+-------------+----------------------+
|        0|            0|                     0|
+---------+-------------+----------------------+



In [22]:
raw_count = tracks_raw.count()
clean_count = tracks_clean.count()

print("Filas originales:", raw_count)
print("Filas limpias:", clean_count)
print("Filas eliminadas:", raw_count - clean_count)
print("Porcentaje eliminado:", round((raw_count - clean_count) / raw_count * 100, 4), "%")

Filas originales: 586672
Filas limpias: 586264
Filas eliminadas: 408
Porcentaje eliminado: 0.0695 %


Guardamos el dataset limpio en parquet

In [23]:
tracks_clean.write.mode("overwrite").parquet("../processed/tracks_clean.parquet")

26/05/03 20:43:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [24]:
tracks_clean_check = spark.read.parquet("../processed/tracks_clean.parquet")

print("Filas en tracks_clean:", tracks_clean_check.count())
tracks_clean_check.printSchema()

Filas en tracks_clean: 586264
root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)



## Preparación base de trabajo 

En esta sección se construye una nueva versión del dataset llamada `tracks_base`, a partir del dataset limpio `tracks_clean`.

El objetivo de esta tabla es conservar las variables originales de las canciones y agregar nuevas variables derivadas que faciliten el análisis exploratorio, la construcción de dashboards y el desarrollo posterior de métricas como el `Hit Score`.

Aunque el dataset original ya contiene variables musicales importantes, algunas de ellas no están en el formato más conveniente para responder las preguntas del proyecto. Por eso, se agregan variables temporales, variables de interpretación más directa y variables categóricas relacionadas con la popularidad.

Las nuevas variables que se crearán son:

- `release_year`: año de lanzamiento de la canción, extraído a partir de `release_date`.
- `release_decade`: década de lanzamiento, útil para analizar tendencias musicales a lo largo del tiempo.
- `duration_min`: duración de la canción en minutos, una medida más interpretable que `duration_ms`.
- `is_hit`: variable binaria que indica si una canción puede considerarse un hit experimentalmente.
- `popularity_class`: categoría de popularidad que clasifica las canciones como de popularidad baja, media o alta.

La variable `is_hit` se define inicialmente usando el umbral `popularity >= 70`. Esta decisión no pretende ser definitiva, sino una primera aproximación académica para identificar canciones altamente populares dentro del dataset. Más adelante, este umbral puede evaluarse usando percentiles de popularidad para determinar si representa adecuadamente a las canciones más destacadas.

La variable `popularity_class` permite simplificar el análisis de la popularidad agrupando las canciones en tres niveles:

- popularidad baja: canciones con `popularity < 40`;
- popularidad media: canciones con `40 <= popularity < 70`;
- popularidad alta: canciones con `popularity >= 70`.

Estas variables derivadas son importantes porque permiten conectar el análisis musical con conceptos de datos masivos: transformación de datos, enriquecimiento de datasets, preparación para análisis agregados, construcción de dashboards y comparación entre grupos de canciones.

Finalmente, esta versión procesada se guardará en formato Parquet para facilitar su reutilización en las siguientes etapas del proyecto.

In [25]:
from pyspark.sql import functions as F

In [26]:
# Cargar dataset limpio
tracks_clean = spark.read.parquet("../processed/tracks_clean.parquet")

In [27]:
# Crear dataset base con variables derivadas
tracks_base = (
    tracks_clean
    .withColumn(
        "release_year",
        F.regexp_extract(F.col("release_date"), r"^(\d{4})", 1).cast("int")
    )
    .withColumn(
        "release_decade",
        (F.floor(F.col("release_year") / 10) * 10).cast("int")
    )
    .withColumn(
        "duration_min",
        F.round(F.col("duration_ms") / 60000, 2)
    )
    .withColumn(
        "is_hit",
        F.when(F.col("popularity") >= 70, 1).otherwise(0)
    )
    .withColumn(
        "popularity_class",
        F.when(F.col("popularity") < 40, "baja")
         .when(F.col("popularity") < 70, "media")
         .otherwise("alta")
    )
)

tracks_base.printSchema()
tracks_base.select(
    "id", "name", "release_date", "release_year", "release_decade",
    "duration_ms", "duration_min", "popularity", "is_hit", "popularity_class"
).show(10)

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release_decade: integer (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- is_hit: integer (nullable = false)
 |-

In [28]:
#verificamos que si aparezca como debe en popularity_class
tracks_base.select("popularity_class").distinct().show(truncate=False)

+----------------+
|popularity_class|
+----------------+
|baja            |
|media           |
|alta            |
+----------------+



In [29]:
tracks_base.select(
    F.count("*").alias("total_filas"),
    F.countDistinct("id").alias("ids_unicos"),
    F.sum(F.col("release_year").isNull().cast("int")).alias("release_year_nulls"),
    F.min("release_year").alias("min_release_year"),
    F.max("release_year").alias("max_release_year"),
    F.min("duration_min").alias("min_duration_min"),
    F.max("duration_min").alias("max_duration_min")
).show()

[Stage 168:============================>                            (4 + 4) / 8]

+-----------+----------+------------------+----------------+----------------+----------------+----------------+
|total_filas|ids_unicos|release_year_nulls|min_release_year|max_release_year|min_duration_min|max_duration_min|
+-----------+----------+------------------+----------------+----------------+----------------+----------------+
|     586264|    586264|                 0|            1900|            2021|            0.26|           93.69|
+-----------+----------+------------------+----------------+----------------+----------------+----------------+



### Distribución de Clases de Popularidad

In [30]:
from pyspark.sql.window import Window

total_window = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

popularity_class_dist = (
    tracks_base
    .groupBy("popularity_class")
    .agg(F.count("*").alias("num_canciones"))
    .withColumn(
        "porcentaje",
        F.round(F.col("num_canciones") / F.sum("num_canciones").over(total_window) * 100, 2)
    )
    .orderBy("popularity_class")
)

popularity_class_dist.show(truncate=False)

26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------------+-------------+----------+
|popularity_class|num_canciones|porcentaje|
+----------------+-------------+----------+
|alta            |7314         |1.25      |
|baja            |428759       |73.13     |
|media           |150191       |25.62     |
+----------------+-------------+----------+



26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Ahora veremos cuantas canciones son **"hits"**

In [31]:
hit_dist = (
    tracks_base
    .groupBy("is_hit")
    .agg(F.count("*").alias("num_canciones"))
    .withColumn(
        "porcentaje",
        F.round(F.col("num_canciones") / F.sum("num_canciones").over(total_window) * 100, 2)
    )
    .orderBy("is_hit")
)

hit_dist.show(truncate=False)

26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+------+-------------+----------+
|is_hit|num_canciones|porcentaje|
+------+-------------+----------+
|0     |578950       |98.75     |
|1     |7314         |1.25      |
+------+-------------+----------+



26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


### Evaluemos si 70 es un buen umbral con percentiles

In [32]:
quantile_probs = [0.50, 0.75, 0.90, 0.95, 0.99]

quantile_values = tracks_base.approxQuantile(
    "popularity",
    quantile_probs,
    0.001
)

percentiles_df = spark.createDataFrame(
    [(f"p{int(p * 100)}", float(v)) for p, v in zip(quantile_probs, quantile_values)],
    ["percentil", "popularity"]
)

percentiles_df.show(truncate=False)

+---------+----------+
|percentil|popularity|
+---------+----------+
|p50      |27.0      |
|p75      |41.0      |
|p90      |52.0      |
|p95      |59.0      |
|p99      |70.0      |
+---------+----------+



inicialmente se propuso usar el umbral `popularity >= 70`.

Para validar esta decisión, se calcularon percentiles aproximados de la variable `popularity`. Los resultados muestran que:

- El percentil 50 es 27, lo que indica que la mitad de las canciones tienen popularidad menor o igual a 27.
- El percentil 75 es 41, por lo que tres cuartas partes del dataset tienen popularidad relativamente baja.
- El percentil 90 es 52.
- El percentil 95 es 59.
- El percentil 99 es 70.

Esto significa que usar `popularity >= 70` define como hits aproximadamente al 1% más popular de las canciones del dataset.

Por lo tanto, el umbral de 70 es una definición estricta, pero adecuada para este proyecto, ya que permite identificar canciones excepcionalmente populares. En el contexto de **HitMaker AI**, esta decisión es útil porque el objetivo no es identificar canciones promedio, sino estudiar patrones asociados con canciones que destacan claramente frente al resto del catálogo.

Sin embargo, esta decisión también implica que la clase `is_hit = 1` será mucho más pequeña que la clase `is_hit = 0`, por lo que se debe considerar el posible desbalance de clases en análisis posteriores.

In [33]:
popularity_bins = (
    tracks_base
    .withColumn(
        "popularity_range",
        F.when(F.col("popularity") < 10, "00-09")
         .when(F.col("popularity") < 20, "10-19")
         .when(F.col("popularity") < 30, "20-29")
         .when(F.col("popularity") < 40, "30-39")
         .when(F.col("popularity") < 50, "40-49")
         .when(F.col("popularity") < 60, "50-59")
         .when(F.col("popularity") < 70, "60-69")
         .when(F.col("popularity") < 80, "70-79")
         .when(F.col("popularity") < 90, "80-89")
         .otherwise("90-100")
    )
    .groupBy("popularity_range")
    .agg(F.count("*").alias("num_canciones"))
    .withColumn(
        "porcentaje",
        F.round(F.col("num_canciones") / F.sum("num_canciones").over(total_window) * 100, 2)
    )
    .orderBy("popularity_range")
)

popularity_bins.show(truncate=False)

26/05/03 20:43:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------------+-------------+----------+
|popularity_range|num_canciones|porcentaje|
+----------------+-------------+----------+
|00-09           |121443       |20.71     |
|10-19           |88240        |15.05     |
|20-29           |110180       |18.79     |
|30-39           |108896       |18.57     |
|40-49           |81748        |13.94     |
|50-59           |47954        |8.18      |
|60-69           |20489        |3.49      |
|70-79           |6361         |1.09      |
|80-89           |904          |0.15      |
|90-100          |49           |0.01      |
+----------------+-------------+----------+



26/05/03 20:43:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/03 20:43:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


### Guardamos tracks_base en Parquet

In [34]:
tracks_base.write.mode("overwrite").parquet("../processed/tracks_base.parquet")

26/05/03 20:43:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

verificamos

In [35]:
tracks_base_check = spark.read.parquet("../processed/tracks_base.parquet")

tracks_base_check.select(
    F.count("*").alias("total_filas"),
    F.countDistinct("id").alias("ids_unicos")
).show()

tracks_base_check.printSchema()

+-----------+----------+
|total_filas|ids_unicos|
+-----------+----------+
|     586264|    586264|
+-----------+----------+

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rel